In [30]:
import pandas as pd
import os
os.getcwd()

'd:\\HomeWork\\DS300\\Đồ án\\github-repository-recommendation'

In [ ]:
from datasets import load_dataset

dataset = load_dataset("ibragim-bad/github-repos-metadata-40M", streaming=True, split="sample")

# 2. Chỉ lấy đúng 10,000 dòng đầu tiên
sub_dataset = dataset.take(10000)

# 3. Chuyển thành Pandas DataFrame (Mất khoảng vài chục giây tùy mạng)
df = pd.DataFrame(sub_dataset)

print(f"Đã load thành công: {len(df)} dòng")

Đã load thành công: 100000 dòng


In [33]:
import os
import time
import requests
import pandas as pd
from urllib.parse import urlparse
from dotenv import load_dotenv
from datasets import load_dataset

# 1. Load biến môi trường
load_dotenv()
GITHUB_TOKEN = os.getenv("CODE_GITHUB")

# 2. Cấu hình file lưu và kích thước Batch
OUTPUT_FILE = "github_metadata_phang.parquet"
CHECKPOINT_BATCH_SIZE = 1000  # Lưu vào ổ cứng cứ sau mỗi 1000 repo

# --- CÁC HÀM XỬ LÝ API ---

def parse_github_url(url):
    path = url.strip().replace("https://github.com/", "").strip("/")
    parts = path.split("/")
    if len(parts) >= 2:
        return parts[0], parts[1]
    raise ValueError("Đường link GitHub không hợp lệ!")

def fetch_repo_metadata_flat(github_url):
    try:
        owner, repo_name = parse_github_url(github_url)
    except ValueError:
        return None
    
    query = """
    query getRepoDetails($owner: String!, $name: String!) {
      repository(owner: $owner, name: $name) {
        owner { login }
        name
        stargazerCount
        forkCount
        watchers { totalCount }
        isFork
        isArchived
        languages(first: 100) {
          totalCount
          edges { size node { name } }
        }
        repositoryTopics(first: 100) {
          totalCount
          nodes { topic { name } }
        }
        diskUsage
        pullRequests { totalCount }
        issues { totalCount }
        description
        primaryLanguage { name }
        createdAt
        pushedAt
        defaultBranchRef { target { ... on Commit { history { totalCount } } } }
        licenseInfo { spdxId }
        assignableUsers { totalCount }
        codeOfConduct { name }
        forkingAllowed
        nameWithOwner
        parent { nameWithOwner }
      }
    }
    """

    headers = {
        "Authorization": f"Bearer {GITHUB_TOKEN}",
        "Content-Type": "application/json"
    }

    try:
        response = requests.post(
            "https://api.github.com/graphql",
            json={"query": query, "variables": {"owner": owner, "name": repo_name}},
            headers=headers,
            timeout=10
        )
    except Exception as e:
        print(f"Lỗi kết nối khi cào {github_url}: {e}")
        return None

    if response.status_code == 403:
        print("\n⚠️ CẢNH BÁO: Đã chạm ngưỡng Rate Limit của GitHub API (5,000 req/h)!")
        return "RATE_LIMITED"

    if response.status_code != 200:
        return None

    data = response.json().get("data", {}).get("repository")
    if not data:
        return None

    # Làm phẳng dữ liệu
    lang_list = [edge["node"]["name"] for edge in data["languages"]["edges"]]
    topic_list = [node["topic"]["name"] for node in data["repositoryTopics"]["nodes"]]

    return {
        "owner": data["owner"]["login"],
        "name": data["name"],
        "stars": data["stargazerCount"],
        "forks": data["forkCount"],
        "watchers": data["watchers"]["totalCount"],
        "isFork": data["isFork"],
        "isArchived": data["isArchived"],
        "languages": ", ".join(lang_list),
        "languageCount": data["languages"]["totalCount"],
        "topics": ", ".join(topic_list),
        "topicCount": data["repositoryTopics"]["totalCount"],
        "diskUsageKb": data["diskUsage"],
        "pullRequests": data["pullRequests"]["totalCount"],
        "issues": data["issues"]["totalCount"],
        "description": data["description"],
        "primaryLanguage": data["primaryLanguage"]["name"] if data["primaryLanguage"] else "",
        "createdAt": data["createdAt"],
        "pushedAt": data["pushedAt"],
        "defaultBranchCommitCount": data["defaultBranchRef"]["target"]["history"]["totalCount"] if data["defaultBranchRef"] else 0,
        "license": data["licenseInfo"]["spdxId"] if data["licenseInfo"] else "",
        "assignableUserCount": data["assignableUsers"]["totalCount"],
        "codeOfConduct": data["codeOfConduct"]["name"] if data["codeOfConduct"] else "",
        "forkingAllowed": data["forkingAllowed"],
        "nameWithOwner": data["nameWithOwner"],
        "parent": data["parent"]["nameWithOwner"] if data["parent"] else ""
    }

In [ ]:
# --- QUY TRÌNH CHÍNH (MAIN WORKFLOW) ---

# 1. Tải danh sách repo từ Hugging Face
print("Đang đọc danh sách repo từ Hugging Face...")
dataset = load_dataset("ibragim-bad/github-repos-metadata-40M", streaming=True, split="sample")
sub_dataset = dataset.take(10000)
df = pd.DataFrame(sub_dataset)

# Tự động tìm tên cột chứa repo name
target_col = 'repo_name' if 'repo_name' in df.columns else 'name'
repo_list = df[target_col].tolist()
print(f"Tổng số repo cần cào: {len(repo_list)}")

# 2. Kiểm tra checkpoint đã có sẵn chưa để bỏ qua các repo đã cào
crawled_repos = set()
if os.path.exists(OUTPUT_FILE):
    try:
        df_existing = pd.read_csv(OUTPUT_FILE, usecols=['nameWithOwner'])
        crawled_repos = set(df_existing['nameWithOwner'].dropna().tolist())
        print(f"Bắt đầu tiếp tục: Đã tìm thấy file local với {len(crawled_repos)} repo đã cào trước đó.")
    except Exception:
        print("Tạo file lưu mới...")

# 3. Tiến hành cào dữ liệu với Checkpoint
batch_results = []
total_saved = len(crawled_repos)

print("\n🚀 Bắt đầu tiến trình cào dữ liệu...")
for idx, repo in enumerate(repo_list, 1):
    # Bỏ qua nếu repo này đã cào thành công ở lần chạy trước
    if repo in crawled_repos:
        continue

    data = fetch_repo_metadata_flat(repo)
    
    if data == "RATE_LIMITED":
        print("Dừng chương trình để tránh mất dữ liệu. Hãy chạy lại sau 1 tiếng!")
        break

    if data:
        batch_results.append(data)
        crawled_repos.add(repo)

    # --- TRONG VÒNG LẶP CHECKPOINT (Thay thế dòng to_csv) ---
    if len(batch_results) >= CHECKPOINT_BATCH_SIZE:
        df_batch = pd.DataFrame(batch_results)
        file_exists = os.path.exists(OUTPUT_FILE)
        
        # Ghi append vào file Parquet bằng engine fastparquet
        df_batch.to_parquet(OUTPUT_FILE, engine='fastparquet', append=file_exists)
        
        total_saved += len(batch_results)
        print(f"✅ [CHECKPOINT] Đã lưu {total_saved}/{len(repo_list)} repo vào '{OUTPUT_FILE}'")

        # Xóa bộ nhớ tạm của batch
        batch_results = []

    # Nghỉ 0.5s giữa các dòng để không làm quá tải kết nối
    time.sleep(0.5)

# Ghi nốt phần dữ liệu dư chưa đủ 1,000 dòng ở cuối
if batch_results:
    df_batch = pd.DataFrame(batch_results)
    file_exists = os.path.exists(OUTPUT_FILE)
    df_batch.to_csv(OUTPUT_FILE, mode='a', index=False, header=not file_exists, encoding="utf-8-sig")
    total_saved += len(batch_results)
    print(f"✅ [HOÀN TẤT BATCH CUỐI] Đã lưu tổng cộng {total_saved} repo.")

print("\n🎉 Tiến trình kết thúc an toàn!")